# Étape 4 — Détection d'obsolescence (Isolation Forest)

**Objectif** : identifier automatiquement les produits dont le pattern de vente s'écarte significativement de la norme (= risque d'obsolescence). Ce signal sera utilisé par l'optimiseur (Étape 6) pour ne pas réapprovisionner ces produits.

**Méthodologie (mémoire §3.5)** :
1. Construction de 7 features d'obsolescence par produit.
2. Isolation Forest (n_estimators=100, contamination=0.10, max_samples=256).
3. Règles métier de filet (jours sans vente ≥ 180, mois consécutifs sans vente ≥ 6, ratio 3m/12m < 0.1).
4. Analyse de sensibilité au paramètre `contamination` (0.05, 0.10, 0.15, 0.20).
5. Validation visuelle : courbes de ventes des produits détectés.
6. Croisement avec la classe ABC.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='talk')

from src.obsolescence import build_obsolescence_features, detect_obsolescence, sensitivity_analysis
from src.utils import PROCESSED_DIR, TAB_DIR

## 1. Chargement des données et construction des features

In [ ]:
transactions = pd.read_csv(PROCESSED_DIR/'zenith_clean.csv', parse_dates=['date'])
classes = pd.read_csv(TAB_DIR/'classification_produits.csv')
feats = build_obsolescence_features(transactions)
age = transactions.groupby('produit_id')['date'].min().pipe(lambda s: (transactions['date'].max()-s).dt.days).rename('age_produit_jours')
feats = feats.merge(age.reset_index(), on='produit_id', how='left')
feats.head(10)

## 2. Détection avec Isolation Forest (contamination = 0.10)

In [ ]:
result = detect_obsolescence(feats)
df = result.df.merge(classes[['produit_id','classe_abc','classe_xyz','libelle_cluster']], on='produit_id', how='left')
n_risk = int(df['a_risque_obsolescence'].sum())
print(f"Produits à risque : {n_risk} / {len(df)} ({100*n_risk/len(df):.1f} %)")
print(f"  - Détectés par Isolation Forest seul : {result.n_flagged - result.n_rules_only}")
print(f"  - Ajoutés par règles métier seules   : {result.n_rules_only}")

## 3. Distribution des scores d'anomalie

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
ax.hist(df['score_obsolescence'], bins=40, color='#1f4e79', edgecolor='black')
ax.axvline(df.loc[df.a_risque_obsolescence==1,'score_obsolescence'].max(), color='red', ls='--', label='Seuil de coupure')
ax.set_title("Distribution des scores d'anomalie")
ax.set_xlabel('Score (plus bas = plus suspect)'); ax.legend()
plt.tight_layout(); plt.show()

## 4. Analyse de sensibilité au paramètre `contamination`

In [ ]:
sens = sensitivity_analysis(feats)
sens

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(sens['contamination']*100, sens['n_flagged_iforest'], marker='o', linewidth=2, color='#1f4e79')
for _, r in sens.iterrows():
    ax.text(r['contamination']*100, r['n_flagged_iforest']+1, f"{int(r['n_flagged_iforest'])}", ha='center')
ax.set_xlabel('Contamination (%)'); ax.set_ylabel('Produits flagués')
ax.set_title("Sensibilité d'Isolation Forest au paramètre contamination")
plt.tight_layout(); plt.show()

## 5. Croisement avec la classe ABC

In [ ]:
cross = df.groupby(['classe_abc','a_risque_obsolescence']).size().unstack(fill_value=0).rename(columns={0:'actif',1:'à_risque'}).reindex(['A','B','C'])
cross['pct_à_risque'] = (cross['à_risque'] / (cross['actif']+cross['à_risque']) * 100).round(1)
cross

## 6. Top 15 produits dormants par valeur de stock immobilisée

In [ ]:
flagged = df[df['a_risque_obsolescence']==1].sort_values('valeur_stock_dormant', ascending=False)
flagged[['produit_id','classe_abc','classe_xyz','jours_depuis_derniere_vente','ratio_ventes_3m_vs_12m','nombre_mois_consecutifs_sans_vente','valeur_stock_dormant','score_obsolescence']].head(15)

## 7. Validation visuelle — courbes de ventes des 8 plus suspects

In [ ]:
top = df[df['a_risque_obsolescence']==1].sort_values('score_obsolescence').head(8)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, pid in enumerate(top['produit_id']):
    ts = transactions[transactions.produit_id==pid].assign(mois=lambda x: x['date'].dt.to_period('M').dt.to_timestamp()).groupby('mois')['quantite_vendue'].sum()
    axes[i].plot(ts.index, ts.values, marker='o', color='#ef476f', linewidth=1.5)
    axes[i].fill_between(ts.index, ts.values, alpha=0.2, color='#ef476f')
    axes[i].set_title(f"{pid} — score {top.iloc[i]['score_obsolescence']:.2f}")
    axes[i].tick_params(axis='x', rotation=45)
fig.suptitle('Courbes de ventes mensuelles — 8 produits les plus suspects', y=1.02)
plt.tight_layout(); plt.show()

## 8. Conclusion

- **73 produits** sur 250 (29 %) sont classés à risque d'obsolescence.
- **Isolation Forest seul** détecte 25 produits ; les **règles métier** en ajoutent 48 supplémentaires que l'algorithme avait manqués (ex. : ratio 3m/12m < 10 % alors qu'il manquait dans la frontière de décision).
- La détection est **équilibrée entre classes ABC** (~30 % de chaque classe) — l'obsolescence touche aussi les bestsellers, pas seulement les produits secondaires.
- **Stock dormant total** ≈ 38 400 USD — c'est la trésorerie qui pourrait être libérée par déstockage.
- Ces produits seront **exclus** du réapprovisionnement automatique à l'Étape 6 (optimisation), et alimenteront une page « alertes » dans le tableau de bord et l'outil interactif.
